In [ ]:
from pycrate_asn1dir import NGAP

'''Values from our known ngap.pcap in test folder'''
PLMN = b"\x00\xf1\x10"      # 001/01
TAC  = b"\x00\x00\x07"      # TAC = 7
SST  = b"\x01"              # sST = 1
GNB_ID_LEN = 22
GNB_ID_VAL = 411
RAN_NAME = "ocucp01"

# Build the NGSetupRequest value using the generated module layout
ngsetup_req_val = (
    "initiatingMessage",
    {
        "procedureCode": NGAP.NGAP_Constants.id_NGSetup._val,
        "criticality": NGAP.NGAP_CommonDataTypes.Criticality._cont_rev[0]
                         if hasattr(NGAP.NGAP_CommonDataTypes.Criticality, "_cont_rev")
                         else "reject",
        "value": (
            "NGSetupRequest",
            {
                "protocolIEs": [
                    {
                        "id": NGAP.NGAP_Constants.id_GlobalRANNodeID._val,
                        "criticality": "reject",
                        "value": (
                            "GlobalRANNodeID",
                            (
                                "globalGNB-ID",
                                {
                                    "pLMNIdentity": PLMN,
                                    "gNB-ID": ("gNB-ID", (GNB_ID_VAL, GNB_ID_LEN)),
                                },
                            ),
                        ),
                    },
                    {
                        "id": NGAP.NGAP_Constants.id_RANNodeName._val,
                        "criticality": "ignore",
                        "value": ("RANNodeName", RAN_NAME),
                    },
                    {
                        "id": NGAP.NGAP_Constants.id_SupportedTAList._val,
                        "criticality": "reject",
                        "value": (
                            "SupportedTAList",
                            [
                                {
                                    "tAC": TAC,
                                    "broadcastPLMNList": [
                                        {
                                            "pLMNIdentity": PLMN,
                                            "tAISliceSupportList": [
                                                {
                                                    "s-NSSAI": {
                                                        "sST": SST
                                                    }
                                                }
                                            ],
                                        }
                                    ],
                                }
                            ],
                        ),
                    },
                    {
                        "id": NGAP.NGAP_Constants.id_DefaultPagingDRX._val,
                        "criticality": "ignore",
                        "value": ("PagingDRX", "v256"),
                    },
                ]
            },
        ),
    },
)

# Use the generated NGAP_PDU object
pdu = NGAP.NGAP_PDU_Descriptions.NGAP_PDU
pdu.set_val(ngsetup_req_val)

encoded = pdu.to_aper()
print("Encoded hex:")
print(encoded.hex())

# Decode back for verification
pdu2 = NGAP.NGAP_PDU_Descriptions.NGAP_PDU
pdu2.from_aper(encoded)

print("\nDecoded structure:")
pdu2()

Encoded hex:
00150032000004001b00080000f1100000066c0052400903006f6375637030310066000d00000000070000f110000000080015400160

Decoded structure:


('initiatingMessage',
 {'procedureCode': 21,
  'criticality': 'reject',
  'value': ('NGSetupRequest',
   {'protocolIEs': [{'id': 27,
      'criticality': 'reject',
      'value': ('GlobalRANNodeID',
       ('globalGNB-ID',
        {'pLMNIdentity': b'\x00\xf1\x10', 'gNB-ID': ('gNB-ID', (411, 22))}))},
     {'id': 82, 'criticality': 'ignore', 'value': ('RANNodeName', 'ocucp01')},
     {'id': 102,
      'criticality': 'reject',
      'value': ('SupportedTAList',
       [{'tAC': b'\x00\x00\x07',
         'broadcastPLMNList': [{'pLMNIdentity': b'\x00\xf1\x10',
           'tAISliceSupportList': [{'s-NSSAI': {'sST': b'\x01'}}]}]}])},
     {'id': 21, 'criticality': 'ignore', 'value': ('PagingDRX', 'v256')}]})})

In [4]:
from pycrate_mobile.NAS import *

Msg, err = parse_NAS_MO(unhexlify(encoded.hex()))

show(Msg)


None


In [ ]:
import socket

hex_data = encoded.hex()
data = bytes.fromhex(hex_data)

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.sendto(data, ("172.31.253.83", 38412))
print("sent")

sent


In [ ]:
'''Needs sudo run separately.'''
# from scapy.all import *
# from scapy.layers.sctp import SCTP, SCTPChunkData

# hex_data = encoded.hex()
# data = bytes.fromhex(hex_data)

# pkt = (
#     IP(src="172.31.253.83", dst="172.31.253.83") /
#     SCTP(sport=38412, dport=38412, tag=0x12345678) /
#     SCTPChunkData(
#         data=data,
#         stream_id=0,
#         stream_seq=0,
#         proto_id=60
#     )
# )

# send(pkt)